# Tiny ImageNet shared training and recovery

Purpose: paired real-data smoke and opt-in pilot execution. Inputs: resolved config, registered protocol and pinned dataset bundle. Outputs: initial controls, resumable states, detached metrics, trace identities and timing summaries. Expected runtime: smoke approximately 1–10 minutes depending on hardware; pilot may exceed 15 minutes and is never started by import. Checkpoints: `checkpoints/<run_id>/<kind>/`, retaining latest two plus best recovery state. Resume: rerun `run_model` with identical config, protocol, data and path; the exact consumed cursor and successful-update schedules resume automatically. Invalid state is refused. No upstream experiment cell is imported.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Paired real-image JEPA execution, successful-update schedules and exact recovery."""
from pathlib import Path
from contextlib import contextmanager
import copy, gc, hashlib, json, math, os, platform, random, shutil, time, uuid
import numpy as np
import torch
from fmm_jepa.models import JEPA
from fmm_jepa.geometry import clear_geometry_cache
from fmm_jepa.training import (atomic_json, digest_file, definition_hashes, config_hash,
    capture_rng, restore_rng, run_writer, parameter_counts, move_masks, write_csv,
    execution_compatibility)

SCHEMA = "fmm-jepa-tiny-checkpoint/v1"

def _hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def _atomic_torch(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with temp.open('wb') as f:
            torch.save(value,f);f.flush();os.fsync(f.fileno())
        os.replace(temp,path)
        fd=os.open(path.parent,os.O_DIRECTORY)
        try:os.fsync(fd)
        finally:os.close(fd)
    finally:temp.unlink(missing_ok=True)

def _rng():
    return capture_rng(torch.Generator().manual_seed(0))

def _restore_rng(state):
    restore_rng(state,torch.Generator())

@contextmanager
def isolated_rng(seed=None):
    """Diagnostics/initialization must not perturb resumed global randomness."""
    state=_rng()
    try:
        if seed is not None:
            random.seed(int(seed));np.random.seed(int(seed)%(2**32));torch.manual_seed(int(seed))
            if torch.cuda.is_available():torch.cuda.manual_seed_all(int(seed))
        yield
    finally:_restore_rng(state)

def training_protocol(config):
    t=config.get('training',{})
    updates=int(t.get('updates',20));micro=int(t.get('microbatch',4));acc=int(t.get('accumulation_steps',2))
    if updates<1 or micro<1 or acc<1:raise ValueError('Positive updates, microbatch and accumulation required')
    if int(config.get('world_size',1))!=1:raise NotImplementedError('Only one device supported')
    if config.get('dtype','float32')!='float32':raise ValueError('Only verified FP32 is enabled')
    if t.get('partial_accumulation','reject')!='reject':raise ValueError('Partial accumulation must be rejected')
    return {'updates':updates,'microbatch':micro,'accumulation_steps':acc,'world_size':1,
      'effective_batch':micro*acc,'optimizer':'AdamW','lr':float(t.get('lr',3e-4)),
      'weight_decay':float(t.get('weight_decay',.05)),
      'weight_decay_exclusions':'All bias parameters and all ndim<=1 parameters (includes normalization scale/bias).',
      'warmup_fraction':float(t.get('warmup_fraction',.05)),
      'schedule':'First ceil(T*warmup_fraction), minimum one, updates linearly warm up; remaining decay cosine to zero at T. Position advances only on successful updates.',
      'ema_start':float(t.get('ema_start',.996)),'ema_end':1.0,
      'ema_schedule':'Cosine interpolation m_start at update1 to 1.0 at updateT; applied only following successful optimizer updates.',
      'gradient_clip':float(t.get('gradient_clip',1.0)),
      'checkpoint_every':min(250,max(1,int(t.get('checkpoint_every',5)))),
      'max_consecutive_failures':int(t.get('max_consecutive_failures',3)),
      'precision':'float32; no autocast, no scaler; geometry/transfers remain float32',
      'partial_accumulation':'reject; complete fixed-size microbatches may cross epoch boundaries',
      'data_order':'Stateless per-epoch permutation from SHA256(data_order_seed,epoch), independent of model/global RNG.',
      'num_workers':0,'initialization':'Fresh models; identical nonmixer context scaffold and complete predictor copied from ViT into FMM, independent storage; teachers recopied.',
      'selection':'final successful-update checkpoint at registered budget; best-loss file retained only for recovery, never selected for primary comparison',
      'failure_policy':'Nonfinite pre-update attempts skipped, no scheduler or EMA advancement; cap consecutive failures; post-update nonfinite state stops without replacing last valid checkpoint.'}

def _model_cfg(config,kind):
    result=copy.deepcopy(config['model']);result['kind']=kind
    return result

def initialize_paired_models(config,artifact_dir):
    """Return fresh independent CPU models, actual counts, and immutable initial controls."""
    directory=Path(artifact_dir)/'initialization';directory.mkdir(parents=True,exist_ok=True)
    identity=_hash({'model':config['model'],'seed':config.get('seeds',{}).get('model_init',20260923)})
    info_path=directory/'manifest.json'
    if info_path.exists():
        info=json.loads(info_path.read_text())
        if info['identity']!=identity:raise ValueError('Existing initialization differs; use a new run ID')
        models={}
        with isolated_rng(int(config.get('seeds',{}).get('model_init',20260923))):
            for kind in ('vit','fmm'):
                for key in ('model','teacher'):
                    record=info['states'][kind][key]
                    if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial state hash mismatch')
                model=JEPA(_model_cfg(config,kind)).cpu()
                model.load_state_dict(torch.load(info['states'][kind]['model']['path'],map_location='cpu',weights_only=True))
                models[kind]=model
        return models,info['parameter_counts'],info
    if list(directory.glob('*.pt')):raise ValueError('Orphan initialization files; refusing silently to replace them')
    with isolated_rng(int(config.get('seeds',{}).get('model_init',20260923))):
        models={kind:JEPA(_model_cfg(config,kind)).cpu() for kind in ('vit','fmm')}
    vit=models['vit'];fmm=models['fmm'];source=vit.context.state_dict();dest=fmm.context.state_dict()
    copied=[]
    for name,tensor in dest.items():
        if '.mixer.' not in name and name in source and source[name].shape==tensor.shape:
            tensor.copy_(source[name]);copied.append('context.'+name)
    fmm.context.load_state_dict(dest)
    fmm.predictor.load_state_dict(vit.predictor.state_dict());copied += ['predictor.'+name for name in vit.predictor.state_dict()]
    for model in models.values():model.target.load_state_dict(model.context.state_dict());model.target.requires_grad_(False).eval()
    for name in copied:
        a=dict(vit.named_parameters()).get(name);b=dict(fmm.named_parameters()).get(name)
        if a is not None and b is not None:
            assert torch.equal(a,b) and a.data_ptr()!=b.data_ptr()
    counts={};states={}
    for kind,model in models.items():
        counts[kind]=parameter_counts(model)
        counts[kind]['teacher_state_bytes']=sum(x.numel()*x.element_size() for x in model.target.state_dict().values())
        states[kind]={}
        for key,state in [('model',model.state_dict()),('teacher',model.target.state_dict())]:
            path=directory/f'{kind}_initial_{key}.pt';_atomic_torch(path,state)
            states[kind][key]={'path':str(path.resolve()),'sha256':digest_file(path)}
    info={'identity':identity,'states':states,'parameter_counts':counts,'copied_components':copied,
          'independent_storage_verified':True,'comparison':'same-scaffold; parameter counts not matched',
          'snapshot_purpose':'immutable initialization controls; not resumable training checkpoints','no_predecessor_weights_loaded':True,'teacher_representation':'final LayerNorm patch outputs averaged over complete grid'}
    atomic_json(info_path,info)
    return models,counts,info

class SuccessfulUpdateSchedule:
    def __init__(self,optimizer,policy):
        self.optimizer=optimizer;self.policy=copy.deepcopy(policy);self.position=0;self._set_next()
    def lr_for(self,step):
        total=self.policy['updates'];warm=max(1,math.ceil(total*self.policy['warmup_fraction']))
        if step<=warm:return self.policy['lr']*step/warm
        return self.policy['lr']*.5*(1+math.cos(math.pi*min(1.,(step-warm)/max(1,total-warm))))
    def _set_next(self):
        for group in self.optimizer.param_groups:group['lr']=self.lr_for(min(self.position+1,self.policy['updates']))
    def step(self):self.position+=1;self._set_next()
    def state_dict(self):return {'position':self.position,'policy':self.policy}
    def load_state_dict(self,value):
        if value['policy']!=self.policy:raise ValueError('Scheduler protocol mismatch')
        self.position=int(value['position']);self._set_next()
    def ema_for(self,step):
        fraction=(step-1)/max(1,self.policy['updates']-1)
        return self.policy['ema_start']+(1-self.policy['ema_start'])*.5*(1-math.cos(math.pi*fraction))

def make_optimizer(model,policy):
    groups={'decay':[],'no_decay':[]};names={'decay':[],'no_decay':[]}
    for name,param in model.named_parameters():
        if not param.requires_grad:continue
        if name.startswith('target.'):raise AssertionError('Teacher must be outside optimizer')
        key='no_decay' if name.endswith('.bias') or param.ndim<=1 else 'decay'
        groups[key].append(param);names[key].append(name)
    optimizer=torch.optim.AdamW([
        {'params':groups['decay'],'weight_decay':policy['weight_decay'],'group_name':'decay'},
        {'params':groups['no_decay'],'weight_decay':0.,'group_name':'no_decay'}],lr=policy['lr'])
    assert sum(p.numel() for g in optimizer.param_groups for p in g['params'])==sum(p.numel() for p in model.parameters() if p.requires_grad)
    return optimizer,SuccessfulUpdateSchedule(optimizer,policy),names

class ConsumedSampler:
    """Cursor advances on delivered examples, never on prefetched samples; workers=0."""
    def __init__(self,ids,seed,cursor=0):
        self.ids=list(ids);self.seed=int(seed);self.cursor=int(cursor);self._epoch=None;self._permutation=None
        if not self.ids or len(self.ids)!=len(set(self.ids)):raise ValueError('Training IDs must be nonempty and unique')
    def take(self,count):
        out=[];occ=[]
        for offset in range(count):
            epoch,position=divmod(self.cursor,len(self.ids))
            if self._epoch!=epoch:
                seed=int(hashlib.sha256(f'{self.seed}:order:{epoch}'.encode()).hexdigest()[:16],16)%(2**63-1)
                self._permutation=torch.randperm(len(self.ids),generator=torch.Generator().manual_seed(seed)).tolist();self._epoch=epoch
            out.append(self.ids[self._permutation[position]]);occ.append(epoch);self.cursor+=1
        return out,occ
    def state_dict(self):return {'cursor':self.cursor,'seed':self.seed,'ids_hash':_hash(self.ids),'num_workers':0,'prefetch':False}
    def load_state_dict(self,value):
        if value['seed']!=self.seed or value['ids_hash']!=_hash(self.ids):raise ValueError('Data-order manifest mismatch')
        self.cursor=int(value['cursor']);self._epoch=None;self._permutation=None

def _identity(config,bundle,protocol):
    return {'config_hash':_hash(config),'protocol_hash':_hash(protocol),
      'dataset_hash':bundle['dataset_manifest']['dataset_fingerprint'],'split_hash':bundle['split_manifest']['split_sha256'],
      'source_hashes':protocol.get('source_hashes',{}),
      'environment':execution_compatibility({**config,'dtype':'float32'})}

def validate_training_manifest(directory,identity=None):
    directory=Path(directory);manifest=json.loads((directory/'manifest.json').read_text())
    if manifest.get('schema')!=SCHEMA:raise ValueError('Training checkpoint schema mismatch')
    if identity is not None and manifest['identity']!=identity:raise ValueError('Configuration/protocol/data/source/environment changed; exact resume refused')
    entries=manifest.get('checkpoints',[])
    if not entries or manifest['latest'] not in {x['file'] for x in entries} or manifest['best'] not in {x['file'] for x in entries}:raise ValueError('Malformed checkpoint manifest')
    for entry in entries:
        if Path(entry['file']).name!=entry['file']:raise ValueError('Invalid checkpoint path')
        if digest_file(directory/entry['file'])!=entry['sha256']:raise ValueError('Training checkpoint hash mismatch: '+entry['file'])
    latest=next(x for x in entries if x['file']==manifest['latest'])
    if latest['successful_updates']!=manifest['successful_updates'] or latest['attempted_updates']!=manifest['attempted_updates']:raise ValueError('Manifest counters disagree')
    if (latest['successful_updates'],latest['attempted_updates'])!=max((x['successful_updates'],x['attempted_updates']) for x in entries):raise ValueError('Latest checkpoint is not the most advanced valid entry')
    return manifest

def _save_state(directory,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,stage):
    directory=Path(directory);old=None
    if (directory/'manifest.json').exists():old=validate_training_manifest(directory,identity)
    filename=f"step_{state['successful_updates']:06d}_attempt_{state['attempted_updates']:06d}.pt"
    if old and filename==old['latest']:return old
    payload={'schema':SCHEMA,'kind':kind,'stage':stage,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
      'scheduler':scheduler.state_dict(),'ema_position':scheduler.position,'scaler':None,'precision':'float32',
      'rng':_rng(),'sampler':sampler.state_dict(),'streams':{'augmentation':'stateless(sample_id,occurrence,seed)',
      'mask':'stateless(sample_id,occurrence,seed)','data_order':'stateless(seed,epoch)','seeds':config.get('seeds',{})},
      'microstep':0,'partial_accumulation':False,'world_size':1,'rank':0,'replay':None,
      'config':copy.deepcopy(config),'protocol':copy.deepcopy(protocol),'identity':identity,'state':copy.deepcopy(state)}
    target=directory/filename;_atomic_torch(target,payload)
    loss=state['metrics'][-1]['loss'] if state['metrics'] else None
    entry={'file':filename,'sha256':digest_file(target),'successful_updates':state['successful_updates'],
           'attempted_updates':state['attempted_updates'],'loss':loss}
    entries=(old['checkpoints'] if old else [])+[entry]
    entries={e['file']:e for e in entries};entries=list(entries.values())
    latest_two=sorted(entries,key=lambda x:(x['successful_updates'],x['attempted_updates']))[-2:]
    best=min(entries,key=lambda x:float('inf') if x['loss'] is None else x['loss'])
    keep={x['file']:x for x in latest_two+[best]}
    manifest={'schema':SCHEMA,'kind':kind,'identity':identity,'latest':filename,'best':best['file'],
       'selection':'final budget checkpoint; best loss retained for recovery only',
       'successful_updates':state['successful_updates'],'attempted_updates':state['attempted_updates'],
       'image_exposures':state['image_exposures'],'cursor':sampler.cursor,
       'resume_notebook':'notebooks/01_TinyImageNet_Debug.ipynb','resume_cell':'train-paired-models',
       'checkpoints':sorted(keep.values(),key=lambda x:(x['successful_updates'],x['attempted_updates']))}
    atomic_json(directory/'manifest.json',manifest)
    for entry in entries:
        if entry['file'] not in keep:(directory/entry['file']).unlink(missing_ok=True)
    return manifest

def _load_state(directory,model,optimizer,scheduler,sampler,identity):
    manifest=validate_training_manifest(directory,identity)
    value=torch.load(Path(directory)/manifest['latest'],map_location='cpu',weights_only=False)
    if value['identity']!=identity or value['schema']!=SCHEMA:raise ValueError('Checkpoint payload identity differs')
    state=value['state']
    if state['successful_updates']!=manifest['successful_updates'] or state['attempted_updates']!=manifest['attempted_updates']:raise ValueError('Checkpoint counter mismatch')
    if value['microstep']!=0 or value['partial_accumulation']:raise ValueError('Partial accumulation checkpoint refused')
    if value['scheduler']['position']!=state['successful_updates'] or value['ema_position']!=state['successful_updates']:raise ValueError('Scheduler/EMA step mismatch')
    if value['sampler']['cursor']!=state['image_exposures']:raise ValueError('Consumed data cursor does not equal image exposures')
    if state['metrics'] and state['metrics'][-1]['successful_updates']!=state['successful_updates']:raise ValueError('Metric/successful update mismatch')
    model.load_state_dict(value['model']);optimizer.load_state_dict(value['optimizer']);scheduler.load_state_dict(value['scheduler'])
    sampler.load_state_dict(value['sampler']);_restore_rng(value['rng'])
    return state,manifest

def pretraining_forward(model,batch):
    """Only augmented pixels and position masks enter JEPA; labels are never read."""
    device=next(model.parameters()).device
    images=batch['images'].to(device=device,dtype=torch.float32)
    masks=move_masks(batch['masks'],device)
    return model(images,masks['context_indices'],masks['context_valid'],masks['target_indices'],masks['target_valid'])

def _finite_output(output):
    tensors=[output['loss'],output['context_features']]+output['prediction']+output['target']
    return all(bool(torch.isfinite(t).all()) for t in tensors)

def _grad_norm(named_parameters,predicate=lambda name:True):
    pieces=[p.grad.detach().float().square().sum() for name,p in named_parameters if predicate(name) and p.grad is not None]
    return float(torch.stack(pieces).sum().sqrt()) if pieces else 0.

def _flush_timing(state,pending,device):
    if pending:
        torch.cuda.synchronize(device)
        for row,begin,end in pending:row['compute_seconds']=row.get('compute_seconds',0.)+begin.elapsed_time(end)/1000.
        pending.clear()
    for row in state['metrics']:
        if 'compute_seconds' in row:
            row['step_seconds']=row['compute_seconds']+row['data_seconds']
            row['images_per_second']=row['effective_batch']/max(row['step_seconds'],1e-12)

def summarize_performance(metrics,kind,device,peak,wall_seconds):
    timed=[x for x in metrics if x['successful_updates']>2 and 'step_seconds' in x]
    def summary(key):
        values=np.asarray([x[key] for x in timed],dtype=float)
        return {'median':float(np.median(values)),'p25':float(np.quantile(values,.25)),'p75':float(np.quantile(values,.75))} if len(values) else None
    return {'model_id':kind,'phase':'pretraining','status':'PASS' if timed else 'NOT_RUN','device':str(device),'dtype':'float32',
      'updates_measured':len(timed),'warmup_updates_excluded':min(2,len(metrics)),
      'complete_jepa_step_seconds':summary('step_seconds'),'jepa_compute_seconds':summary('compute_seconds'),
      'data_preprocessing_seconds':summary('data_seconds'),'throughput_images_per_second':summary('images_per_second'),
      'peak_cuda_allocated_bytes':int(peak) if str(device).startswith('cuda') else None,
      'peak_definition':'max_memory_allocated of this model training after other model moved to CPU; excludes CUDA context and cached reserved memory',
      'wall_seconds_including_checkpoints_and_diagnostics':float(wall_seconds),
      'timing_scope':'Actual accumulated forward/backward, clipping, optimizer, scheduler and EMA; data measured separately and included in complete step. CUDA events flushed at checkpoints; diagnostics/checkpoint I/O excluded from step timing.',
      'n_grid':metrics[0]['n_grid'] if metrics else None,'n_visible_mean':float(np.mean([x['visible_tokens_mean'] for x in metrics])) if metrics else None}

def _training_performance_history(previous,state,kind,device,peak,wall,resumed,identity):
    """Aggregate measured invocations without substituting reload peaks for training peaks."""
    identity_digest=_hash(identity)
    if previous is not None and previous.get('identity_hash')!=identity_digest:
        raise ValueError('Historical performance identity changed; refusing to overwrite evidence')
    history=copy.deepcopy(previous.get('invocation_history',[])) if previous else []
    final=state['successful_updates'];updated=final>resumed
    invocation={'start_successful_update':resumed,'end_successful_update':final,
      'successful_updates_executed':final-resumed,'status':'MEASURED_TRAINING' if updated else 'NO_TRAINING_REUSED_CHECKPOINT',
      'wall_seconds':float(wall),'peak_cuda_allocated_bytes':int(peak) if updated and str(device).startswith('cuda') else None,
      'load_only_peak_cuda_allocated_bytes':int(peak) if not updated and str(device).startswith('cuda') else None,
      'scope':'This invocation only; wall includes checkpoint/diagnostic work and excludes model construction.'}
    history.append(invocation)
    measured=[r for r in history if r['successful_updates_executed']>0]
    covered=0
    for record in sorted(measured,key=lambda r:r['start_successful_update']):
        if record['start_successful_update']>covered:break
        covered=max(covered,record['end_successful_update'])
    coverage_complete=covered>=final and final>0
    known_peak=[r['peak_cuda_allocated_bytes'] for r in measured if r['peak_cuda_allocated_bytes'] is not None]
    performance=summarize_performance(state['metrics'],kind,device,max(known_peak,default=0),sum(r['wall_seconds'] for r in measured))
    if not coverage_complete:
        performance['peak_cuda_allocated_bytes']=None
        performance['wall_seconds_including_checkpoints_and_diagnostics']=None
    performance.update(identity_hash=identity_digest,invocation_history=history,last_invocation=invocation,
      measurement_reused=not updated,successful_update_coverage_complete=coverage_complete,
      history_covered_through_update=covered,final_successful_update=final,
      timing_coverage='Per-update timings come from all authoritative checkpoint metrics; run-wide wall/peak require recorded invocations covering every successful update.',
      historical_wall_peak_status='COMPLETE_RECORDED_HISTORY' if coverage_complete else 'UNKNOWN_EARLIER_INVOCATIONS',
      peak_definition='Maximum of measured training-invocation max_memory_allocated values with geometry caches cleared before/after each model. Includes that invocation model/input/optimizer/geometry/diagnostic allocations and shared live backend workspaces; excludes reserved cache/CUDA context. Completed reload-only peaks are kept separately and never replace training peaks.')
    return performance


def run_model(kind,config,dataset_bundle,artifact_dir,checkpoint_dir,protocol,model=None,stop_after=None,diagnostic_callback=None):
    """Shared smoke/pilot path. stop_after bounds this invocation, never the schedule budget."""
    from fmm_jepa.tiny_imagenet import make_batch
    if kind not in ('vit','fmm'):raise ValueError('Unknown model kind')
    policy=training_protocol(config);device=torch.device(config.get('device','cpu'))
    identity=_identity(config,dataset_bundle,protocol)
    artifact_dir=Path(artifact_dir);checkpoint_dir=Path(checkpoint_dir)
    artifact_dir.mkdir(parents=True,exist_ok=True);checkpoint_dir.mkdir(parents=True,exist_ok=True)
    performance_path=artifact_dir/f'{kind}_performance.json'
    previous_performance=json.loads(performance_path.read_text()) if performance_path.exists() else None
    if previous_performance is not None and previous_performance.get('identity_hash')!=_hash(identity):
        raise ValueError('Historical performance identity mismatch; use a new run ID')
    clear_geometry_cache();gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    if model is None:model=initialize_paired_models(config,artifact_dir)[0][kind]
    model=model.to(device=device,dtype=torch.float32)
    optimizer,scheduler,group_names=make_optimizer(model,policy)
    atomic_json(artifact_dir/f'{kind}_optimizer_groups.json',group_names)
    sampler=ConsumedSampler(dataset_bundle['train_ids'],config.get('seeds',{}).get('data_order',17))
    state={'successful_updates':0,'attempted_updates':0,'image_exposures':0,'consecutive_failures':0,
           'metrics':[],'traces':[],'failures':[],'diagnostics':[]}
    start=time.perf_counter();pending=[];resumed=0;last_save=time.monotonic();manifest=None
    stop=policy['updates'] if stop_after is None else min(int(stop_after),policy['updates'])
    if stop<0:raise ValueError('stop_after cannot be negative')
    if device.type=='cuda':gc.collect();torch.cuda.empty_cache();torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
    diagnostic_steps={0,policy['updates']//2,policy['updates']}
    def diagnostics():
        step=state['successful_updates']
        if diagnostic_callback and step in diagnostic_steps and step not in {r['step'] for r in state['diagnostics']}:
            with isolated_rng():
                record=diagnostic_callback(model,kind,step)
            state['diagnostics'].append({'step':step,'model_id':kind,'results':record})
    with run_writer(checkpoint_dir):
        if (checkpoint_dir/'manifest.json').exists():
            state,manifest=_load_state(checkpoint_dir,model,optimizer,scheduler,sampler,identity);resumed=state['successful_updates']
        elif list(checkpoint_dir.glob('*.pt')):raise ValueError('Orphan checkpoint files without manifest; refusing restart')
        diagnostics()
        if manifest is None:manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'initial')
        try:
            while state['successful_updates']<stop:
                model.train();optimizer.zero_grad(set_to_none=True)
                state['attempted_updates']+=1;attempt=state['attempted_updates'];before_epoch=sampler.cursor//len(sampler.ids)
                values=[];trace={'model_id':kind,'attempted_update':attempt,'sample_ids':[],'occurrences':[],'augmentation_ids':[],'mask_ids':[]}
                data_seconds=0.;compute_wall=0.;visible=[];target_counts=[];micro_events=[]
                lr=float(optimizer.param_groups[0]['lr']);step=state['successful_updates']+1
                try:
                    for micro in range(policy['accumulation_steps']):
                        data_start=time.perf_counter();ids,occurrences=sampler.take(policy['microbatch'])
                        batch=make_batch(dataset_bundle['dataset'],ids,occurrences,config,training=True)
                        if batch['images'].shape[0]!=policy['microbatch']:raise ValueError('Partial microbatch refused')
                        state['image_exposures']+=len(ids)
                        # Labels intentionally never enter the JEPA call.
                        trace['sample_ids'].extend(ids);trace['occurrences'].extend(occurrences)
                        trace['augmentation_ids'].extend(batch['augmentation_ids']);trace['mask_ids'].extend(batch['mask_ids'])
                        images=batch['images'].to(device=device,dtype=torch.float32)
                        masks=move_masks(batch['masks'],device)
                        data_seconds+=time.perf_counter()-data_start
                        if device.type=='cuda':
                            event_start=torch.cuda.Event(enable_timing=True);event_end=torch.cuda.Event(enable_timing=True);event_start.record()
                        t0=time.perf_counter()
                        output=pretraining_forward(model,{'images':images,'masks':masks})
                        if not _finite_output(output):raise FloatingPointError('Nonfinite activation/loss before optimizer')
                        (output['loss']/policy['accumulation_steps']).backward()
                        values.append({k:float(v) for k,v in output['metrics'].items()})
                        visible.extend(masks['context_valid'].sum(1).detach().cpu().tolist())
                        target_counts.extend(torch.stack([v.sum(1) for v in masks['target_valid']],1).detach().cpu().flatten().tolist())
                        if device.type=='cuda':event_end.record();micro_events.append((event_start,event_end))
                        compute_wall+=time.perf_counter()-t0
                        del output,images,masks,batch
                    update_start=time.perf_counter()
                    if device.type=='cuda':update_event_start=torch.cuda.Event(enable_timing=True);update_event_end=torch.cuda.Event(enable_timing=True);update_event_start.record()
                    grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
                    if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):raise FloatingPointError('Nonfinite/absent gradients before optimizer')
                    if any(p.grad is not None for p in model.target.parameters()):raise AssertionError('Teacher gradient leakage')
                except FloatingPointError as exc:
                    optimizer.zero_grad(set_to_none=True);state['consecutive_failures']+=1
                    trace['successful']=False;trace['failure']=str(exc);state['traces'].append(trace)
                    state['failures'].append({'attempt':attempt,'reason':str(exc),'consumed_cursor':sampler.cursor})
                    if state['consecutive_failures']>=policy['max_consecutive_failures']:raise RuntimeError('Repeated nonfinite attempts exceeded cap') from exc
                    continue
                grad_norm=float(torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],policy['gradient_clip'],error_if_nonfinite=True))
                # Branch norms are measured after clipping; global norm above is before clipping.
                near_grad=_grad_norm(model.context.named_parameters(),lambda n:'.near_' in n)
                far_grad=_grad_norm(model.context.named_parameters(),lambda n:'.far_' in n or '.translation' in n)
                optimizer.step()
                if not all(bool(torch.isfinite(p).all()) for p in model.parameters()):raise RuntimeError('Nonfinite updated state; last valid checkpoint retained, offending state not published')
                momentum=scheduler.ema_for(step);model.update_teacher(momentum);scheduler.step()
                if device.type=='cuda':update_event_end.record();micro_events.append((update_event_start,update_event_end))
                compute_wall+=time.perf_counter()-update_start
                state['successful_updates']=step;state['consecutive_failures']=0
                row={'model_id':kind,'seed':int(config.get('seeds',{}).get('model_init',20260923)),
                  'phase':'pretraining','split':'train','n_grid':(int(config.get('image_size',64))//int(config['model']['patch_size']))**2,'successful_updates':step,'attempted_updates':attempt,
                  'image_exposures':state['image_exposures'],'consumed_cursor':sampler.cursor,
                  'loss':float(np.mean([x['loss'] for x in values])),'lr':lr,'ema_momentum':momentum,
                  'scheduler_position':scheduler.position,'gradient_norm_preclip':grad_norm,'clipped':grad_norm>policy['gradient_clip'],
                  'near_gradient_norm_postclip':near_grad,'far_gradient_norm_postclip':far_grad,
                  'visible_tokens_mean':float(np.mean(visible)),'visible_tokens_min':min(visible),'visible_tokens_max':max(visible),
                  'target_tokens_mean':float(np.mean(target_counts)),'effective_batch':policy['effective_batch'],
                  'microbatch':policy['microbatch'],'accumulation_steps':policy['accumulation_steps'],'data_seconds':data_seconds,
                  'teacher_gradients_absent':True,'teacher_eval':not model.target.training,
                  'failed_attempts_total':len(state['failures'])}
                for name in values[0]:
                    if name!='loss':row[name]=float(np.mean([x[name] for x in values]))
                if device.type=='cuda':
                    # Aggregate event pairs only at checkpoint boundaries to avoid synchronizing each normal update.
                    row['_cuda_event_count']=len(micro_events)
                    for ev0,ev1 in micro_events:pending.append((row,ev0,ev1))
                else:row['compute_seconds']=compute_wall
                state['metrics'].append(row);trace['successful']=True;trace['successful_update']=step;state['traces'].append(trace)
                diagnostics()
                epoch_boundary=sampler.cursor//len(sampler.ids)!=before_epoch
                if step%policy['checkpoint_every']==0 or step==stop or epoch_boundary or time.monotonic()-last_save>=900:
                    _flush_timing(state,pending,device)
                    manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'update_boundary')
                    last_save=time.monotonic()
        except BaseException as exc:
            # Save only a finite completed optimizer boundary; an in-flight partial update is recovered from the previous checkpoint.
            _flush_timing(state,pending,device)
            atomic_json(artifact_dir/f'{kind}_interruption.json',{'exception':repr(exc),'successful_updates':state['successful_updates'],
               'attempted_updates':state['attempted_updates'],'image_exposures':state['image_exposures'],
               'resume_from':str(checkpoint_dir/'manifest.json'),'partial_attempt_replayed_from_last_valid_boundary':True})
            model.cpu();optimizer.zero_grad(set_to_none=True);clear_geometry_cache();gc.collect()
            if device.type=='cuda':torch.cuda.empty_cache()
            raise
        _flush_timing(state,pending,device)
        manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'invocation_end')
    peak=torch.cuda.max_memory_allocated(device) if device.type=='cuda' else 0
    wall=time.perf_counter()-start
    performance=_training_performance_history(previous_performance,state,kind,device,peak,wall,resumed,identity)
    if state['metrics']:write_csv(artifact_dir/f'{kind}_training_metrics.csv',state['metrics'])
    atomic_json(artifact_dir/f'{kind}_training_traces.json',state['traces'])
    atomic_json(artifact_dir/f'{kind}_training_diagnostics.json',state['diagnostics'])
    atomic_json(artifact_dir/f'{kind}_performance.json',performance)
    model=model.cpu();optimizer.zero_grad(set_to_none=True);del optimizer,scheduler;clear_geometry_cache();gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    return {'model':model,'metrics':state['metrics'],'traces':state['traces'],'diagnostics':state['diagnostics'],
      'performance':performance,'checkpoint_manifest':manifest,'checkpoint_dir':str(checkpoint_dir),
      'resume':{'resumed_from_step':resumed,'final_step':state['successful_updates'],'configured_updates':policy['updates'],
         'complete':state['successful_updates']==policy['updates'],'actual_image_exposures':state['image_exposures']}}


The module imports definitions only. Notebook 01 registers protocol identities, verifies exact CPU recovery on real images, and then invokes this shared path sequentially for ViT and FMM. Both initial teachers are retained for frozen-feature controls.